# Lab 2: Learning When Labels Are Scarce

**Semi-supervised network intrusion detection on CICIDS2017**

This is the submission-safe revision of the Lab 2 experiment. It contains:

- the original **Lab 1 Random Forest** as the required full-label reference;
- few-label lower baselines at **1%, 5%, and 10%**;
- iterative **pseudo-labelling** and **co-training**;
- a validation-only confidence-threshold ablation;
- accuracy, macro-F1, attack recall, ROC-AUC, and false-alarm rate (FAR);
- three fixed random seeds, pseudo-label quality diagnostics, one results table, and a label-budget curve.

> **Why this notebook has no stored results:** the previous outputs were produced from a new raw-CSV split (2,830,743 rows, 70 features), not the Lab 1 split. They were therefore invalid for the assignment and have deliberately been removed. This notebook has no raw-data or synthetic fallback: it stops unless the exact Lab 1 split is present and passes the checks below.


## 1. Environment and reproducibility

Run from a fresh kernel. The portable scikit-learn backend and a fixed thread limit prevent silent model changes between CPU-only and GPU machines.


In [1]:
import os
from pathlib import Path

SEED = 42
CPU_THREADS = 4
for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[variable] = str(CPU_THREADS)

import gc
import hashlib
import importlib.util
import json
import platform
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils.class_weight import compute_sample_weight
from threadpoolctl import threadpool_limits

np.random.seed(SEED)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

print(f"Python {sys.version.split()[0]} | scikit-learn {sklearn.__version__} | pandas {pd.__version__}")
print(f"Fixed numerical-library thread limit: {CPU_THREADS}")
print(f"Platform: {platform.platform()}")


Python 3.13.7 | scikit-learn 1.9.1 | pandas 3.0.5
Fixed numerical-library thread limit: 4
Platform: Linux-6.17.0-41-generic-x86_64-with-glibc2.42


## 2. Configuration

The experiment is intentionally strict:

- only `data/lab1_splits.npz` is accepted;
- the expected Lab 1 row counts, 68-feature schema, attack rate, and test class counts are checked;
- the full Lab 1 reference comes from Table 1 of the supplied Lab 1 report;
- `HistGradientBoostingClassifier` is fixed for every new Lab 2 model;
- seeds 42, 43, and 44 are averaged.

If the source repository is laid out differently, change only `LAB1_METRICS_CANDIDATES`; do not change the split checks to make an unrelated dataset pass.


In [2]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "lab2_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_MODE = "npz"
LAB1_SPLITS = DATA_DIR / "lab1_splits.npz"
EXPECTED_LAB1_SHAPES = {
    "train": (267_984, 68),
    "validation": (89_328, 68),
    "test": (89_329, 68),
}
EXPECTED_TEST_CLASS_COUNTS = {0: 75_868, 1: 13_461}
EXPECTED_ATTACK_RATE = 0.1507
ATTACK_RATE_TOLERANCE = 0.01

# Optional strongest check. After independently verifying the exported Lab 1
# file once, paste the printed digest here and keep it unchanged for all runs.
EXPECTED_LAB1_SHA256 = None

BUDGETS = (0.01, 0.05, 0.10)
SEEDS = (42, 43, 44)
CONFIDENCE_THRESHOLD = 0.95
SSL_ROUNDS = 2
PSEUDO_WEIGHT = 0.50
PSEUDO_TO_TRUE_RATIO_PER_ROUND = 1.0
MAX_PSEUDO_PER_CLASS_PER_ROUND = 1_000_000
PREDICTION_BATCH_SIZE = 65_536

RUN_ABLATION = True
ABLATION_BUDGET = 0.01
ABLATION_THRESHOLDS = (0.70, 0.80, 0.90, 0.95, 0.99)

MODEL_BACKEND = "sklearn"
HGB_ITERATIONS = 120
RUN_SAME_MODEL_FULL_LABEL = True

# Required full-label upper reference, copied from Lab 1 report Table 1.
LAB1_REFERENCE = {
    "model": "Random Forest (Lab 1)",
    "accuracy": 0.9984,
    "macro_f1": 0.9968,
    "recall_attack": 0.9923,
    "roc_auc": 0.9999,
    "FAR": 0.0006,
    "source": "Lab 1 report, Table 1",
}

# Reuse Lab 1's FAR helper when the submitted repository is present.
LAB1_METRICS_CANDIDATES = (
    PROJECT_DIR / "collab" / "collab" / "src" / "metrics.py",
    PROJECT_DIR / "src" / "metrics.py",
)
REQUIRE_LAB1_METRICS_REUSE = True

print("Configuration loaded.")
print("Seeds:", SEEDS, "| model backend:", MODEL_BACKEND)


Configuration loaded.
Seeds: (42, 43, 44) | model backend: sklearn


## 3. Exact Lab 1 data

Export the already-cleaned, already-split Lab 1 arrays once. The `.npz` file must contain `X_train`, `y_train`, `X_val`, `y_val`, `X_test`, `y_test`, and `feature_names`. Save `feature_names` as a normal Unicode string array so the file can be opened with `allow_pickle=False`.

Example, run in the Lab 1 environment and adapt the dictionary keys if necessary:

```python
from pathlib import Path
import joblib
import numpy as np

source = Path("collab/collab/data/processed/splits.joblib")
split = joblib.load(source)
Path("data").mkdir(exist_ok=True)
np.savez_compressed(
    "data/lab1_splits.npz",
    X_train=np.asarray(split["X_train"]),
    y_train=np.asarray(split["yb_train"]),
    X_val=np.asarray(split["X_val"]),
    y_val=np.asarray(split["yb_val"]),
    X_test=np.asarray(split["X_test"]),
    y_test=np.asarray(split["yb_test"]),
    feature_names=np.asarray(split["feature_names"], dtype=str),
)
```

The test set remains fully labelled and is never used to choose thresholds or hyperparameters.


In [3]:
REQUIRED_NPZ_KEYS = {
    "X_train", "y_train", "X_val", "y_val", "X_test", "y_test", "feature_names"
}


def _validate_binary_target(name, target, expected_rows):
    target = np.asarray(target).reshape(-1)
    if len(target) != expected_rows:
        raise AssertionError(f"{name} has {len(target):,} labels; expected {expected_rows:,}.")
    if not np.isfinite(target).all():
        raise AssertionError(f"{name} contains non-finite labels.")
    unique = set(np.unique(target).tolist())
    if unique != {0, 1}:
        raise AssertionError(f"{name} must use binary labels 0=benign and 1=attack; found {sorted(unique)}.")
    return np.asarray(target, dtype=np.int8)


def _split_fingerprint(named_arrays, feature_names):
    digest = hashlib.sha256()
    for name, array in named_arrays:
        contiguous = np.ascontiguousarray(array)
        digest.update(name.encode("utf-8"))
        digest.update(str(contiguous.dtype).encode("ascii"))
        digest.update(np.asarray(contiguous.shape, dtype="<i8").tobytes())
        digest.update(memoryview(contiguous).cast("B"))
    digest.update("\0".join(map(str, feature_names)).encode("utf-8"))
    return digest.hexdigest()


def load_exact_lab1_splits():
    if DATA_MODE != "npz":
        raise AssertionError("Submission configuration requires DATA_MODE='npz'.")
    if not LAB1_SPLITS.is_file():
        raise FileNotFoundError(
            f"Missing {LAB1_SPLITS}. Export the exact Lab 1 split as described above. "
            "Raw CICIDS CSVs and synthetic data are deliberately not accepted."
        )

    with np.load(LAB1_SPLITS, allow_pickle=False) as data:
        missing = REQUIRED_NPZ_KEYS.difference(data.files)
        if missing:
            raise KeyError(f"{LAB1_SPLITS} is missing {sorted(missing)}.")
        X_train = np.ascontiguousarray(data["X_train"], dtype=np.float32)
        X_val = np.ascontiguousarray(data["X_val"], dtype=np.float32)
        X_test = np.ascontiguousarray(data["X_test"], dtype=np.float32)
        y_train = _validate_binary_target("y_train", data["y_train"], len(X_train))
        y_val = _validate_binary_target("y_val", data["y_val"], len(X_val))
        y_test = _validate_binary_target("y_test", data["y_test"], len(X_test))
        feature_names = np.asarray(data["feature_names"]).astype(str).reshape(-1)

    arrays = {"train": X_train, "validation": X_val, "test": X_test}
    for name, array in arrays.items():
        expected = EXPECTED_LAB1_SHAPES[name]
        if array.shape != expected:
            raise AssertionError(
                f"{name} shape is {array.shape}; expected exact Lab 1 shape {expected}. "
                "Do not regenerate the split in this notebook."
            )
        if not np.isfinite(array).all():
            raise AssertionError(f"{name} contains NaN or infinity; the Lab 1 cleaning removed those rows.")

    if len(feature_names) != X_train.shape[1] or len(set(feature_names.tolist())) != len(feature_names):
        raise AssertionError("feature_names must contain one unique name per feature column.")

    observed_test_counts = dict(zip(*np.unique(y_test, return_counts=True)))
    observed_test_counts = {int(key): int(value) for key, value in observed_test_counts.items()}
    if observed_test_counts != EXPECTED_TEST_CLASS_COUNTS:
        raise AssertionError(
            f"Test class counts are {observed_test_counts}; expected {EXPECTED_TEST_CLASS_COUNTS}."
        )

    total_attacks = int(y_train.sum() + y_val.sum() + y_test.sum())
    total_rows = len(y_train) + len(y_val) + len(y_test)
    overall_attack_rate = total_attacks / total_rows
    if abs(overall_attack_rate - EXPECTED_ATTACK_RATE) > ATTACK_RATE_TOLERANCE:
        raise AssertionError(
            f"Attack rate {overall_attack_rate:.4f} does not match Lab 1 ({EXPECTED_ATTACK_RATE:.4f})."
        )

    fingerprint = _split_fingerprint(
        [
            ("X_train", X_train), ("y_train", y_train),
            ("X_val", X_val), ("y_val", y_val),
            ("X_test", X_test), ("y_test", y_test),
        ],
        feature_names,
    )
    if EXPECTED_LAB1_SHA256 is not None and fingerprint != EXPECTED_LAB1_SHA256:
        raise AssertionError(
            f"Lab 1 split digest changed: {fingerprint}; expected {EXPECTED_LAB1_SHA256}."
        )

    return X_train, X_val, X_test, y_train, y_val, y_test, feature_names, fingerprint


In [4]:
X_train, X_val, X_test, y_train, y_val, y_test, feature_names, LAB1_SPLIT_SHA256 = load_exact_lab1_splits()
ACTIVE_DATA_MODE = "lab1_npz"

split_summary = pd.DataFrame(
    {
        "rows": [len(y_train), len(y_val), len(y_test)],
        "features": [X_train.shape[1], X_val.shape[1], X_test.shape[1]],
        "benign": [(y_train == 0).sum(), (y_val == 0).sum(), (y_test == 0).sum()],
        "attack": [(y_train == 1).sum(), (y_val == 1).sum(), (y_test == 1).sum()],
        "attack_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
    },
    index=["train", "validation", "test"],
)
memory_mb = sum(array.nbytes for array in (X_train, X_val, X_test)) / (1024 ** 2)
print(split_summary.to_string(float_format=lambda value: f"{value:.4f}"))
print(f"Feature-array memory: {memory_mb:.1f} MiB ({X_train.dtype})")
print("Verified Lab 1 split SHA-256:", LAB1_SPLIT_SHA256)


              rows  features  benign  attack  attack_rate
train       267984        68  227599   40385       0.1507
validation   89328        68   75867   13461       0.1507
test         89329        68   75868   13461       0.1507
Feature-array memory: 115.9 MiB (float32)
Verified Lab 1 split SHA-256: 1891044e6bb39ea93549cd7b80f70cd83801efbbe59c69280a0ed90230d8f5ac


## 4. Fixed model backend

Every newly trained model uses the same scikit-learn `HistGradientBoostingClassifier`. There is no automatic CUDA/XGBoost fallback, so two machines do not silently run different algorithms. The original Lab 1 Random Forest remains a separate, explicitly sourced reference.


In [5]:
if MODEL_BACKEND != "sklearn":
    raise AssertionError("This submission fixes MODEL_BACKEND='sklearn' for reproducibility.")
ACTIVE_BACKEND = "sklearn_hist_gradient_boosting"
print("Active model backend:", ACTIVE_BACKEND)


def make_estimator(seed):
    return HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=HGB_ITERATIONS,
        max_leaf_nodes=31,
        min_samples_leaf=5,
        l2_regularization=1.0,
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=10,
        random_state=seed,
    )


def fit_estimator(X, y, seed, trust_weights=None):
    y = np.asarray(y, dtype=np.int8)
    if np.unique(y).size != 2:
        raise ValueError("Every fitted subset must contain both benign and attack samples.")
    balanced = compute_sample_weight(class_weight="balanced", y=y).astype(np.float32)
    if trust_weights is not None:
        balanced *= np.asarray(trust_weights, dtype=np.float32)
    balanced /= balanced.mean()
    model = make_estimator(seed)
    with threadpool_limits(limits=CPU_THREADS):
        model.fit(X, y, sample_weight=balanced)
    return model


def predict_proba_batched(model, X, batch_size=PREDICTION_BATCH_SIZE):
    batches = []
    for start in range(0, len(X), batch_size):
        batch = np.asarray(model.predict_proba(X[start:start + batch_size]), dtype=np.float32)
        batches.append(batch)
    return np.concatenate(batches, axis=0)


def predict_proba_indexed(model, X, row_indices, feature_indices=None, batch_size=PREDICTION_BATCH_SIZE):
    batches = []
    for start in range(0, len(row_indices), batch_size):
        rows = row_indices[start:start + batch_size]
        batch = X[rows] if feature_indices is None else X[np.ix_(rows, feature_indices)]
        batches.append(np.asarray(model.predict_proba(batch), dtype=np.float32))
    return np.concatenate(batches, axis=0)


Active model backend:

 sklearn_hist_gradient_boosting


## 5. Lab 1 metric reuse and label-budget splits

Class `0` is benign and class `1` is attack. FAR is therefore `FP / (FP + TN)`. This section imports Lab 1's `false_alarm_rate` helper and checks it against the confusion-matrix definition. The stratified budget splitter hides labels only inside the Lab 1 training set.


In [6]:
def load_lab1_false_alarm_rate():
    for path in LAB1_METRICS_CANDIDATES:
        if not path.is_file():
            continue
        try:
            if str(path.parent) not in sys.path:
                sys.path.insert(0, str(path.parent))
            spec = importlib.util.spec_from_file_location("lab1_metrics_reused", path)
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            helper = getattr(module, "false_alarm_rate")
            return helper, path
        except Exception as error:
            raise RuntimeError(f"Could not import Lab 1 metrics from {path}: {error}") from error
    if REQUIRE_LAB1_METRICS_REUSE:
        candidates = "\n".join(f"  - {path}" for path in LAB1_METRICS_CANDIDATES)
        raise FileNotFoundError(
            "Lab 1 metrics.py was not found. Keep it in the submitted repository or update "
            f"LAB1_METRICS_CANDIDATES. Checked:\n{candidates}"
        )
    return None, None


LAB1_FALSE_ALARM_RATE, LAB1_METRICS_PATH = load_lab1_false_alarm_rate()
LAB1_METRICS_REUSED = LAB1_FALSE_ALARM_RATE is not None
print("Lab 1 metrics reuse:", LAB1_METRICS_PATH if LAB1_METRICS_REUSED else "disabled")

METRIC_COLUMNS = ["accuracy", "macro_f1", "recall_attack", "roc_auc", "FAR"]


def evaluate_model(model, X, y):
    probability = predict_proba_batched(model, X)
    prediction = np.argmax(probability, axis=1).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    local_far = fp / (fp + tn) if (fp + tn) else np.nan
    if LAB1_FALSE_ALARM_RATE is not None:
        reused_far = float(LAB1_FALSE_ALARM_RATE(y, prediction))
        if not np.isclose(reused_far, local_far, rtol=0.0, atol=1e-12):
            raise AssertionError(
                f"Lab 1 FAR helper returned {reused_far}; confusion-matrix FAR is {local_far}."
            )
        far = reused_far
    else:
        far = local_far
    auc = roc_auc_score(y, probability[:, 1]) if np.unique(y).size == 2 else np.nan
    return {
        "accuracy": accuracy_score(y, prediction),
        "macro_f1": f1_score(y, prediction, average="macro", zero_division=0),
        "recall_attack": recall_score(y, prediction, pos_label=1, zero_division=0),
        "roc_auc": auc,
        "FAR": far,
    }


def make_label_budget_split(y, budget, seed):
    requested = max(int(round(len(y) * budget)), 4 * np.unique(y).size)
    if requested >= len(y):
        raise ValueError("The label budget leaves no unlabelled pool.")
    splitter = StratifiedShuffleSplit(n_splits=1, train_size=requested, random_state=seed)
    labelled, unlabelled = next(splitter.split(np.zeros(len(y)), y))
    labelled = labelled.astype(np.int64)
    unlabelled = unlabelled.astype(np.int64)
    if np.intersect1d(labelled, unlabelled).size:
        raise AssertionError("Labelled and unlabelled indices overlap.")
    if len(labelled) + len(unlabelled) != len(y):
        raise AssertionError("The budget split does not cover the training set.")
    if set(np.unique(y[labelled])) != {0, 1}:
        raise AssertionError("The labelled slice lost a class.")
    return labelled, unlabelled


def choose_confident(probability, threshold, per_class_cap=None):
    prediction = np.argmax(probability, axis=1).astype(np.int8)
    confidence = np.max(probability, axis=1)
    passed_mask = confidence >= threshold
    chosen_parts = []
    diagnostics = {"passed_threshold": int(passed_mask.sum())}

    for class_id, class_name in ((0, "benign"), (1, "attack")):
        positions = np.flatnonzero(passed_mask & (prediction == class_id))
        diagnostics[f"passed_{class_name}"] = int(len(positions))
        if per_class_cap is not None and len(positions) > per_class_cap:
            order = np.lexsort((positions, -confidence[positions]))
            positions = positions[order[:per_class_cap]]
        diagnostics[f"accepted_{class_name}"] = int(len(positions))
        chosen_parts.append(positions)

    chosen = np.concatenate(chosen_parts) if chosen_parts else np.empty(0, dtype=np.int64)
    if len(chosen):
        order = np.lexsort((chosen, -confidence[chosen]))
        chosen = chosen[order]
    diagnostics["accepted_after_cap"] = int(len(chosen))
    return chosen, prediction[chosen], confidence[chosen], diagnostics


def weighted_history_mean(history, value_column, weight_column):
    if history.empty or value_column not in history:
        return np.nan
    valid = history[value_column].notna() & (history[weight_column] > 0)
    if not valid.any():
        return np.nan
    return float(np.average(history.loc[valid, value_column], weights=history.loc[valid, weight_column]))


Lab 1 metrics reuse: /home/steven/Desktop/AI-for-Cybersecurity-Lab2/collab/collab/src/metrics.py


## 6. Required method: iterative pseudo-labelling

At each round, the model predicts the remaining unlabelled pool. The main experiment keeps the original per-class safety cap and logs both the number that passed the threshold and the number retained after the cap. The hidden training labels are consulted only **after selection** to audit pseudo-label precision; that diagnostic never affects selection, fitting, or stopping.


In [7]:
def train_pseudo_labeller(
    X,
    labelled_indices,
    unlabelled_indices,
    labelled_targets,
    seed,
    threshold=CONFIDENCE_THRESHOLD,
    initial_model=None,
    apply_class_cap=True,
    audit_targets=None,
):
    fit_indices = labelled_indices.copy()
    fit_targets = np.asarray(labelled_targets, dtype=np.int8).copy()
    trust = np.ones(len(fit_indices), dtype=np.float32)
    remaining = unlabelled_indices.copy()
    model = initial_model
    history = []
    total_added = 0

    per_class_cap = None
    if apply_class_cap:
        per_class_cap = min(
            MAX_PSEUDO_PER_CLASS_PER_ROUND,
            max(1, int(np.ceil(len(labelled_indices) * PSEUDO_TO_TRUE_RATIO_PER_ROUND / 2))),
        )

    for round_index in range(SSL_ROUNDS):
        if model is None or round_index > 0:
            model = fit_estimator(X[fit_indices], fit_targets, seed + round_index, trust)

        probability = predict_proba_indexed(model, X, remaining)
        positions, pseudo_targets, confidence, selection = choose_confident(
            probability, threshold, per_class_cap
        )
        selected = remaining[positions]

        precision = np.nan
        precision_benign = np.nan
        precision_attack = np.nan
        if audit_targets is not None and len(selected):
            truth = np.asarray(audit_targets)[selected]
            precision = float(np.mean(truth == pseudo_targets))
            for class_id, key in ((0, "benign"), (1, "attack")):
                mask = pseudo_targets == class_id
                value = float(np.mean(truth[mask] == pseudo_targets[mask])) if mask.any() else np.nan
                if key == "benign":
                    precision_benign = value
                else:
                    precision_attack = value

        history.append(
            {
                "round": round_index + 1,
                "threshold": threshold,
                "class_cap": per_class_cap,
                "remaining_before": len(remaining),
                **selection,
                "mean_confidence": float(confidence.mean()) if len(confidence) else np.nan,
                "pseudo_label_precision": precision,
                "pseudo_precision_benign": precision_benign,
                "pseudo_precision_attack": precision_attack,
            }
        )
        if len(selected) == 0:
            break

        fit_indices = np.concatenate([fit_indices, selected])
        fit_targets = np.concatenate([fit_targets, pseudo_targets])
        trust = np.concatenate([trust, np.full(len(selected), PSEUDO_WEIGHT, dtype=np.float32)])
        keep = np.ones(len(remaining), dtype=bool)
        keep[positions] = False
        remaining = remaining[keep]
        total_added += len(selected)
        if len(remaining) == 0:
            break

    if total_added:
        model = fit_estimator(X[fit_indices], fit_targets, seed + SSL_ROUNDS, trust)
    return model, pd.DataFrame(history), total_added


## 7. Second method: co-training

Features are deterministically shuffled into two non-overlapping views. Each learner teaches only the other learner; conflicting proposals are rejected. Diagnostics report pre-cap proposals, conflicts, transfer precision, and each view's stand-alone validation score. Final predictions average the two learners.


In [8]:
class CoTrainingEnsemble:
    def __init__(self, model_a, model_b, view_a, view_b):
        self.model_a = model_a
        self.model_b = model_b
        self.view_a = np.asarray(view_a, dtype=np.int64)
        self.view_b = np.asarray(view_b, dtype=np.int64)
        self.classes_ = np.asarray([0, 1], dtype=np.int8)

    def predict_proba(self, X):
        probability_a = np.asarray(self.model_a.predict_proba(X[:, self.view_a]), dtype=np.float32)
        probability_b = np.asarray(self.model_b.predict_proba(X[:, self.view_b]), dtype=np.float32)
        return (probability_a + probability_b) / 2.0


def make_feature_views(n_features, seed):
    if n_features < 2:
        raise ValueError("Co-training needs at least two features.")
    permutation = np.random.default_rng(seed).permutation(n_features)
    midpoint = (n_features + 1) // 2
    return np.sort(permutation[:midpoint]), np.sort(permutation[midpoint:])


def _audit_precision(audit_targets, global_indices, proposed_labels):
    if audit_targets is None or len(global_indices) == 0:
        return np.nan
    return float(np.mean(np.asarray(audit_targets)[global_indices] == proposed_labels))


def train_co_trainer(
    X,
    labelled_indices,
    unlabelled_indices,
    labelled_targets,
    seed,
    threshold=CONFIDENCE_THRESHOLD,
    audit_targets=None,
):
    view_a, view_b = make_feature_views(X.shape[1], seed)
    indices_a = labelled_indices.copy()
    indices_b = labelled_indices.copy()
    targets_a = np.asarray(labelled_targets, dtype=np.int8).copy()
    targets_b = np.asarray(labelled_targets, dtype=np.int8).copy()
    trust_a = np.ones(len(indices_a), dtype=np.float32)
    trust_b = np.ones(len(indices_b), dtype=np.float32)
    remaining = unlabelled_indices.copy()
    history = []
    total_unique_added = 0
    model_a = model_b = None
    added_after_fit = False

    per_class_cap = min(
        MAX_PSEUDO_PER_CLASS_PER_ROUND,
        max(1, int(np.ceil(len(labelled_indices) * PSEUDO_TO_TRUE_RATIO_PER_ROUND / 2))),
    )

    for round_index in range(SSL_ROUNDS):
        model_a = fit_estimator(
            X[np.ix_(indices_a, view_a)], targets_a, seed + 10 * round_index, trust_a
        )
        model_b = fit_estimator(
            X[np.ix_(indices_b, view_b)], targets_b, seed + 10 * round_index + 1, trust_b
        )
        added_after_fit = False

        probability_a = predict_proba_indexed(model_a, X, remaining, view_a)
        probability_b = predict_proba_indexed(model_b, X, remaining, view_b)
        positions_a, labels_a, confidence_a, diagnostic_a = choose_confident(
            probability_a, threshold, per_class_cap
        )
        positions_b, labels_b, confidence_b, diagnostic_b = choose_confident(
            probability_b, threshold, per_class_cap
        )

        proposals_a = {int(position): int(label) for position, label in zip(positions_a, labels_a)}
        proposals_b = {int(position): int(label) for position, label in zip(positions_b, labels_b)}
        conflicts = {
            position for position in proposals_a.keys() & proposals_b.keys()
            if proposals_a[position] != proposals_b[position]
        }
        accepted_a_positions = np.asarray(
            [position for position in positions_a if int(position) not in conflicts], dtype=np.int64
        )
        accepted_b_positions = np.asarray(
            [position for position in positions_b if int(position) not in conflicts], dtype=np.int64
        )
        accepted_a_labels = np.asarray(
            [proposals_a[int(position)] for position in accepted_a_positions], dtype=np.int8
        )
        accepted_b_labels = np.asarray(
            [proposals_b[int(position)] for position in accepted_b_positions], dtype=np.int8
        )

        # A teaches B; B teaches A.
        if len(accepted_b_positions):
            taught_indices = remaining[accepted_b_positions]
            indices_a = np.concatenate([indices_a, taught_indices])
            targets_a = np.concatenate([targets_a, accepted_b_labels])
            trust_a = np.concatenate(
                [trust_a, np.full(len(taught_indices), PSEUDO_WEIGHT, dtype=np.float32)]
            )
        if len(accepted_a_positions):
            taught_indices = remaining[accepted_a_positions]
            indices_b = np.concatenate([indices_b, taught_indices])
            targets_b = np.concatenate([targets_b, accepted_a_labels])
            trust_b = np.concatenate(
                [trust_b, np.full(len(taught_indices), PSEUDO_WEIGHT, dtype=np.float32)]
            )

        accepted_union = np.union1d(accepted_a_positions, accepted_b_positions)
        union_labels = np.asarray(
            [
                proposals_a[int(position)] if int(position) in proposals_a else proposals_b[int(position)]
                for position in accepted_union
            ],
            dtype=np.int8,
        )
        union_counts = np.bincount(union_labels, minlength=2) if len(union_labels) else np.zeros(2, dtype=int)

        history.append(
            {
                "round": round_index + 1,
                "threshold": threshold,
                "class_cap": per_class_cap,
                "remaining_before": len(remaining),
                "learner_a_passed_threshold": diagnostic_a["passed_threshold"],
                "learner_b_passed_threshold": diagnostic_b["passed_threshold"],
                "learner_a_accepted_after_cap": diagnostic_a["accepted_after_cap"],
                "learner_b_accepted_after_cap": diagnostic_b["accepted_after_cap"],
                "conflicts_rejected": len(conflicts),
                "accepted_unique": len(accepted_union),
                "accepted_unique_benign": int(union_counts[0]),
                "accepted_unique_attack": int(union_counts[1]),
                "mean_confidence_a": float(confidence_a.mean()) if len(confidence_a) else np.nan,
                "mean_confidence_b": float(confidence_b.mean()) if len(confidence_b) else np.nan,
                "precision_a_to_b": _audit_precision(
                    audit_targets, remaining[accepted_a_positions], accepted_a_labels
                ),
                "precision_b_to_a": _audit_precision(
                    audit_targets, remaining[accepted_b_positions], accepted_b_labels
                ),
                "pseudo_label_precision": _audit_precision(
                    audit_targets, remaining[accepted_union], union_labels
                ),
            }
        )
        if len(accepted_union) == 0:
            break

        keep = np.ones(len(remaining), dtype=bool)
        keep[accepted_union] = False
        remaining = remaining[keep]
        total_unique_added += len(accepted_union)
        added_after_fit = True
        if len(remaining) == 0:
            break

    if added_after_fit:
        model_a = fit_estimator(X[np.ix_(indices_a, view_a)], targets_a, seed + 10_000, trust_a)
        model_b = fit_estimator(X[np.ix_(indices_b, view_b)], targets_b, seed + 10_001, trust_b)
    ensemble = CoTrainingEnsemble(model_a, model_b, view_a, view_b)
    return ensemble, pd.DataFrame(history), total_unique_added


## 8. Full-label references

The required upper reference is the original Lab 1 Random Forest result. The optional 100%-label histogram model is trained only to provide a like-for-like ceiling for the new Lab 2 learner; it does **not** replace or masquerade as the Lab 1 result.


In [9]:
result_rows = []
full_same_model_metrics_by_seed = {}
few_model_cache = {}
split_cache = {}
ssl_histories = {}
co_view_rows = []

lab1_reference_table = pd.DataFrame([LAB1_REFERENCE])
print("Required Lab 1 full-label reference:")
print(lab1_reference_table.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

if RUN_SAME_MODEL_FULL_LABEL:
    for seed in SEEDS:
        started = time.perf_counter()
        full_model = fit_estimator(X_train, y_train, seed)
        metrics = evaluate_model(full_model, X_test, y_test)
        elapsed = time.perf_counter() - started
        full_same_model_metrics_by_seed[seed] = metrics
        result_rows.append(
            {
                "seed": seed,
                "budget_pct": 100.0,
                "method": "Full-label same-model ceiling",
                "true_labels": len(y_train),
                "pseudo_labels": 0,
                "pseudo_label_precision": np.nan,
                "runtime_s": elapsed,
                **metrics,
            }
        )
        print(
            f"Same-model full-label seed {seed}: macro-F1={metrics['macro_f1']:.4f}, "
            f"FAR={metrics['FAR']:.4f}, {elapsed:.1f}s"
        )
        del full_model
        gc.collect()


Required Lab 1 full-label reference:
                model  accuracy  macro_f1  recall_attack  roc_auc    FAR                source
Random Forest (Lab 1)    0.9984    0.9968         0.9923   0.9999 0.0006 Lab 1 report, Table 1


Same-model full-label seed 42: macro-F1=0.9959, FAR=0.0022, 266.2s


Same-model full-label seed 43: macro-F1=0.9966, FAR=0.0018, 305.2s


Same-model full-label seed 44: macro-F1=0.9962, FAR=0.0020, 436.8s


## 9. Few-label, pseudo-labelling, and co-training runs

For each seed and budget, all three methods receive the same labelled/unlabelled split. Main results are evaluated on the untouched test set only after the design is fixed. Co-training view diagnostics use validation data and do not tune the model.


In [10]:
for seed in SEEDS:
    for budget in BUDGETS:
        labelled, unlabelled = make_label_budget_split(y_train, budget, seed)
        split_cache[(seed, budget)] = (labelled, unlabelled)
        labelled_targets = y_train[labelled]
        actual_budget = 100.0 * len(labelled) / len(y_train)
        print(
            f"\nSeed {seed} | requested {100 * budget:.0f}% | "
            f"actual {actual_budget:.3f}% ({len(labelled):,} true labels)"
        )

        started = time.perf_counter()
        few_model = fit_estimator(X_train[labelled], labelled_targets, seed)
        metrics = evaluate_model(few_model, X_test, y_test)
        elapsed = time.perf_counter() - started
        few_model_cache[(seed, budget)] = few_model
        result_rows.append(
            {
                "seed": seed,
                "budget_pct": 100 * budget,
                "method": "Few-label lower baseline",
                "true_labels": len(labelled),
                "pseudo_labels": 0,
                "pseudo_label_precision": np.nan,
                "runtime_s": elapsed,
                **metrics,
            }
        )
        print(f"  Few-label : macro-F1={metrics['macro_f1']:.4f}, FAR={metrics['FAR']:.4f}, {elapsed:.1f}s")

        started = time.perf_counter()
        pseudo_model, pseudo_history, pseudo_count = train_pseudo_labeller(
            X_train,
            labelled,
            unlabelled,
            labelled_targets,
            seed,
            threshold=CONFIDENCE_THRESHOLD,
            initial_model=few_model,
            apply_class_cap=True,
            audit_targets=y_train,
        )
        metrics = evaluate_model(pseudo_model, X_test, y_test)
        elapsed = time.perf_counter() - started
        pseudo_precision = weighted_history_mean(
            pseudo_history, "pseudo_label_precision", "accepted_after_cap"
        )
        ssl_histories[(seed, budget, "Pseudo-labelling")] = pseudo_history
        result_rows.append(
            {
                "seed": seed,
                "budget_pct": 100 * budget,
                "method": "Pseudo-labelling",
                "true_labels": len(labelled),
                "pseudo_labels": pseudo_count,
                "pseudo_label_precision": pseudo_precision,
                "runtime_s": elapsed,
                **metrics,
            }
        )
        print(
            f"  Pseudo     : macro-F1={metrics['macro_f1']:.4f}, FAR={metrics['FAR']:.4f}, "
            f"+{pseudo_count:,} pseudo, precision={pseudo_precision:.4f}, {elapsed:.1f}s"
        )
        del pseudo_model
        gc.collect()

        started = time.perf_counter()
        co_model, co_history, co_count = train_co_trainer(
            X_train,
            labelled,
            unlabelled,
            labelled_targets,
            seed,
            threshold=CONFIDENCE_THRESHOLD,
            audit_targets=y_train,
        )
        metrics = evaluate_model(co_model, X_test, y_test)
        elapsed = time.perf_counter() - started
        co_precision = weighted_history_mean(
            co_history, "pseudo_label_precision", "accepted_unique"
        )
        ssl_histories[(seed, budget, "Co-training")] = co_history
        result_rows.append(
            {
                "seed": seed,
                "budget_pct": 100 * budget,
                "method": "Co-training",
                "true_labels": len(labelled),
                "pseudo_labels": co_count,
                "pseudo_label_precision": co_precision,
                "runtime_s": elapsed,
                **metrics,
            }
        )

        view_a_metrics = evaluate_model(co_model.model_a, X_val[:, co_model.view_a], y_val)
        view_b_metrics = evaluate_model(co_model.model_b, X_val[:, co_model.view_b], y_val)
        for view_name, view_metrics in (("view_a", view_a_metrics), ("view_b", view_b_metrics)):
            co_view_rows.append(
                {"seed": seed, "budget_pct": 100 * budget, "view": view_name, **view_metrics}
            )
        print(
            f"  Co-training: macro-F1={metrics['macro_f1']:.4f}, FAR={metrics['FAR']:.4f}, "
            f"+{co_count:,} pseudo, precision={co_precision:.4f}, {elapsed:.1f}s"
        )
        del co_model
        gc.collect()



Seed 42 | requested 1% | actual 1.000% (2,680 true labels)


  Few-label : macro-F1=0.9822, FAR=0.0029, 59.2s


  Pseudo     : macro-F1=0.9786, FAR=0.0036, +5,360 pseudo, precision=1.0000, 162.3s


  Co-training: macro-F1=0.9779, FAR=0.0024, +7,533 pseudo, precision=0.9927, 173.0s



Seed 42 | requested 5% | actual 5.000% (13,399 true labels)


...

  Co-training: macro-F1=0.9952, FAR=0.0010, +44,184 pseudo, precision=1.0000, 191.6s



Seed 44 | requested 10% | actual 10.000% (26,798 true labels)


  Few-label : macro-F1=0.9945, FAR=0.0023, 33.2s


  Pseudo     : macro-F1=0.9954, FAR=0.0013, +53,596 pseudo, precision=1.0000, 407.7s


  Co-training: macro-F1=0.9952, FAR=0.0009, +66,874 pseudo, precision=1.0000, 245.4s


## 10. Main results table

**Table 1.** Test-set performance by label budget. Lab 2 values are means across seeds 42, 43, and 44; `macro_f1_std` is the sample standard deviation. `macro_f1_gain` compares each SSL method with the same-seed-budget lower baseline before averaging. The final row is the original Lab 1 Random Forest reference, not a retrained substitute.


In [11]:
raw_results = pd.DataFrame(result_rows)

lower_by_seed_budget = (
    raw_results[raw_results["method"] == "Few-label lower baseline"]
    .set_index(["seed", "budget_pct"])["macro_f1"]
)
raw_results["macro_f1_gain"] = raw_results.apply(
    lambda row: row["macro_f1"] - lower_by_seed_budget.get((row["seed"], row["budget_pct"]), np.nan)
    if row["method"] in {"Pseudo-labelling", "Co-training"}
    else np.nan,
    axis=1,
)

aggregation = {metric: "mean" for metric in METRIC_COLUMNS}
aggregation.update(
    {
        "true_labels": "mean",
        "pseudo_labels": "mean",
        "pseudo_label_precision": "mean",
        "macro_f1_gain": "mean",
        "runtime_s": "mean",
    }
)
results = raw_results.groupby(["budget_pct", "method"], as_index=False).agg(aggregation)
f1_std = (
    raw_results.groupby(["budget_pct", "method"])["macro_f1"]
    .std(ddof=1)
    .rename("macro_f1_std")
    .reset_index()
)
results = results.merge(f1_std, on=["budget_pct", "method"], how="left")

method_order = {
    "Few-label lower baseline": 0,
    "Pseudo-labelling": 1,
    "Co-training": 2,
    "Full-label same-model ceiling": 3,
}
results["_order"] = results["method"].map(method_order)
results = results.sort_values(["budget_pct", "_order"]).drop(columns="_order").reset_index(drop=True)

lab1_row = {
    "budget_pct": 100.0,
    "method": "Lab 1 full-label Random Forest",
    "true_labels": len(y_train),
    "pseudo_labels": 0,
    "pseudo_label_precision": np.nan,
    "accuracy": LAB1_REFERENCE["accuracy"],
    "macro_f1": LAB1_REFERENCE["macro_f1"],
    "macro_f1_std": np.nan,
    "macro_f1_gain": np.nan,
    "recall_attack": LAB1_REFERENCE["recall_attack"],
    "roc_auc": LAB1_REFERENCE["roc_auc"],
    "FAR": LAB1_REFERENCE["FAR"],
    "runtime_s": np.nan,
}
results_with_lab1 = pd.concat([results, pd.DataFrame([lab1_row])], ignore_index=True)

display_columns = [
    "budget_pct", "method", "true_labels", "pseudo_labels", "pseudo_label_precision",
    "accuracy", "macro_f1", "macro_f1_std", "macro_f1_gain",
    "recall_attack", "roc_auc", "FAR", "runtime_s",
]
print(results_with_lab1[display_columns].to_string(index=False, float_format=lambda value: f"{value:.4f}"))

raw_results.to_csv(OUTPUT_DIR / "lab2_results_raw.csv", index=False)
results_with_lab1.to_csv(OUTPUT_DIR / "lab2_results_summary.csv", index=False)
lab1_reference_table.to_csv(OUTPUT_DIR / "lab1_full_label_reference.csv", index=False)
print(f"\nSaved result tables in {OUTPUT_DIR}")


 budget_pct                         method  true_labels  pseudo_labels  pseudo_label_precision  accuracy  macro_f1  macro_f1_std  macro_f1_gain  recall_attack  roc_auc    FAR  runtime_s
     1.0000       Few-label lower baseline    2680.0000         0.0000                     NaN    0.9896    0.9795        0.0029            NaN         0.9566   0.9954 0.0045    37.7960
     1.0000               Pseudo-labelling    2680.0000      5360.0000                  1.0000    0.9906    0.9814        0.0029         0.0018         0.9572   0.9962 0.0035   151.2248
     1.0000                    Co-training    2680.0000      7802.0000                  0.9973    0.9916    0.9833        0.0047         0.0038         0.9561   0.9974 0.0021   171.0785
     5.0000       Few-label lower baseline   13399.0000         0.0000                     NaN    0.9969    0.9940        0.0001            NaN         0.9894   0.9995 0.0017    33.3762
     5.0000               Pseudo-labelling   13399.0000     26800.0000

## 11. Change-one-thing experiment: confidence threshold

The ablation runs at the scarcest (1%) budget and uses thresholds 0.70, 0.80, 0.90, 0.95, and 0.99. The per-class cap is disabled **for every ablation run**, so it cannot mask the threshold effect. Between after-rows, the threshold is the only changed setting. Selection and model choice use validation—not test—metrics.

**Table 2.** One-percent-label validation performance before pseudo-labelling and after each threshold. `passed_threshold` and pseudo-label precision make the coverage/quality trade-off visible.


In [ ]:
ablation_results = pd.DataFrame()
if RUN_ABLATION:
    ablation_seed = SEEDS[0]
    labelled, unlabelled = split_cache.get(
        (ablation_seed, ABLATION_BUDGET),
        make_label_budget_split(y_train, ABLATION_BUDGET, ablation_seed),
    )
    labelled_targets = y_train[labelled]
    base_model = few_model_cache.get((ablation_seed, ABLATION_BUDGET))
    if base_model is None:
        base_model = fit_estimator(X_train[labelled], labelled_targets, ablation_seed)
    base_metrics = evaluate_model(base_model, X_val, y_val)

    rows = [
        {
            "setting": "Before: few-label baseline",
            "threshold": np.nan,
            "passed_threshold": 0,
            "pseudo_labels": 0,
            "pseudo_label_precision": np.nan,
            "runtime_s": 0.0,
            **base_metrics,
        }
    ]
    for threshold in ABLATION_THRESHOLDS:
        started = time.perf_counter()
        model, history, pseudo_count = train_pseudo_labeller(
            X_train,
            labelled,
            unlabelled,
            labelled_targets,
            ablation_seed,
            threshold=threshold,
            initial_model=base_model,
            apply_class_cap=False,
            audit_targets=y_train,
        )
        metrics = evaluate_model(model, X_val, y_val)
        rows.append(
            {
                "setting": f"After: threshold={threshold:.2f}",
                "threshold": threshold,
                "passed_threshold": int(history["passed_threshold"].sum()),
                "pseudo_labels": pseudo_count,
                "pseudo_label_precision": weighted_history_mean(
                    history, "pseudo_label_precision", "accepted_after_cap"
                ),
                "runtime_s": time.perf_counter() - started,
                **metrics,
            }
        )
        del model
        gc.collect()

    ablation_results = pd.DataFrame(rows)
    ablation_results["macro_f1_change"] = ablation_results["macro_f1"] - base_metrics["macro_f1"]
    columns = [
        "setting", "threshold", "passed_threshold", "pseudo_labels", "pseudo_label_precision",
        "accuracy", "macro_f1", "macro_f1_change", "recall_attack", "roc_auc", "FAR",
    ]
    print(ablation_results[columns].to_string(index=False, float_format=lambda value: f"{value:.4f}"))
    ablation_results.to_csv(OUTPUT_DIR / "confidence_ablation_validation.csv", index=False)
    print("Ablation decisions used validation metrics only; test labels were not evaluated here.")
else:
    print("Ablation skipped because RUN_ABLATION=False.")


## 12. Label-budget curve

**Figure 1.** Mean test macro-F1 against the labelled fraction of the Lab 1 training set. The budget-dependent few-label curve is the lower baseline. The dashed green line is the required original Lab 1 Random Forest reference. The dotted purple line is an additional same-model, 100%-label ceiling and is not labelled as Lab 1. Shading shows one standard deviation across the three seeds.


In [ ]:
figure, axis = plt.subplots(figsize=(9, 5.5))
colours = {
    "Few-label lower baseline": "#555555",
    "Pseudo-labelling": "#1f77b4",
    "Co-training": "#e67e22",
}
markers = {"Few-label lower baseline": "o", "Pseudo-labelling": "s", "Co-training": "^"}

plotted_values = []
for method in ("Few-label lower baseline", "Pseudo-labelling", "Co-training"):
    subset = results[results["method"] == method].sort_values("budget_pct")
    x_values = subset["budget_pct"].to_numpy(dtype=float)
    y_values = subset["macro_f1"].to_numpy(dtype=float)
    spread = subset["macro_f1_std"].fillna(0).to_numpy(dtype=float)
    plotted_values.extend(y_values.tolist())
    axis.plot(
        x_values,
        y_values,
        label=method,
        color=colours[method],
        marker=markers[method],
        linewidth=2.2,
        markersize=7,
    )
    axis.fill_between(
        x_values,
        np.clip(y_values - spread, 0, 1),
        np.clip(y_values + spread, 0, 1),
        color=colours[method],
        alpha=0.15,
    )

lab1_f1 = float(LAB1_REFERENCE["macro_f1"])
plotted_values.append(lab1_f1)
axis.axhline(
    lab1_f1,
    color="#2ca02c",
    linestyle="--",
    linewidth=2.2,
    label=f"Lab 1 full-label Random Forest ({lab1_f1:.4f})",
)

if RUN_SAME_MODEL_FULL_LABEL and full_same_model_metrics_by_seed:
    same_model_f1 = float(np.mean([item["macro_f1"] for item in full_same_model_metrics_by_seed.values()]))
    plotted_values.append(same_model_f1)
    axis.axhline(
        same_model_f1,
        color="#9467bd",
        linestyle=":",
        linewidth=2.2,
        label=f"Full-label same-model ceiling ({same_model_f1:.4f})",
    )

axis.set(
    xlabel="Labelled fraction of Lab 1 training set (%)",
    ylabel="Macro-F1 on untouched Lab 1 test set",
    title="Semi-supervised performance as labels become available",
    xticks=[100 * budget for budget in BUDGETS],
    ylim=(max(0.0, min(plotted_values) - 0.08), min(1.01, max(plotted_values) + 0.02)),
)
axis.grid(True, alpha=0.25)
axis.legend(loc="best", frameon=True)
figure.tight_layout()
figure_path = OUTPUT_DIR / "macro_f1_vs_label_budget.png"
figure.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved Figure 1 to {figure_path}")


## 13. SSL diagnostics

This table is for failure analysis, not model selection. It records how many unlabelled rows cleared the threshold before the main-experiment cap, how many were accepted, and how often the guessed labels matched the hidden audit labels. Co-training's two stand-alone validation views are saved separately.


In [ ]:
history_frames = []
for (seed, budget, method), history in ssl_histories.items():
    history_frames.append(
        history.assign(seed=seed, budget_pct=100 * budget, method=method)
    )
ssl_diagnostics = pd.concat(history_frames, ignore_index=True, sort=False)
co_view_diagnostics = pd.DataFrame(co_view_rows)

diagnostic_columns = [
    column for column in (
        "seed", "budget_pct", "method", "round", "threshold", "class_cap",
        "remaining_before", "passed_threshold", "accepted_after_cap",
        "learner_a_passed_threshold", "learner_b_passed_threshold",
        "learner_a_accepted_after_cap", "learner_b_accepted_after_cap",
        "conflicts_rejected", "accepted_unique", "pseudo_label_precision",
    ) if column in ssl_diagnostics.columns
]
print(ssl_diagnostics[diagnostic_columns].to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("\nCo-training stand-alone validation views:")
print(co_view_diagnostics.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

ssl_diagnostics.to_csv(OUTPUT_DIR / "ssl_diagnostics.csv", index=False)
co_view_diagnostics.to_csv(OUTPUT_DIR / "co_training_view_validation.csv", index=False)


## 14. Automated checks and run manifest

These checks fail for the wrong data source, wrong Lab 1 dimensions/class counts, a silently changed backend, insufficient seeds, incomplete methods, invalid metrics, broken label-budget partitions, or an ablation in which the cap still controls acceptance.


In [ ]:
if ACTIVE_DATA_MODE != "lab1_npz":
    raise AssertionError("Only the exact Lab 1 NPZ split is valid for submission.")
if ACTIVE_BACKEND != "sklearn_hist_gradient_boosting" or MODEL_BACKEND != "sklearn":
    raise AssertionError("The model backend changed from the fixed scikit-learn implementation.")
if len(SEEDS) < 3 or tuple(SEEDS) != (42, 43, 44):
    raise AssertionError("Final results must average fixed seeds (42, 43, 44).")
if REQUIRE_LAB1_METRICS_REUSE and not LAB1_METRICS_REUSED:
    raise AssertionError("Lab 1 metric reuse is required but was not active.")

for name, array in (("train", X_train), ("validation", X_val), ("test", X_test)):
    if array.shape != EXPECTED_LAB1_SHAPES[name]:
        raise AssertionError(f"{name} no longer matches the verified Lab 1 shape.")
if {int(key): int(value) for key, value in zip(*np.unique(y_test, return_counts=True))} != EXPECTED_TEST_CLASS_COUNTS:
    raise AssertionError("The test class counts no longer match Lab 1.")

expected_methods = {"Few-label lower baseline", "Pseudo-labelling", "Co-training"}
for seed in SEEDS:
    for budget in BUDGETS:
        observed = set(
            raw_results.loc[
                (raw_results["seed"] == seed)
                & np.isclose(raw_results["budget_pct"], 100 * budget),
                "method",
            ]
        )
        if observed != expected_methods:
            raise AssertionError(
                f"Seed {seed}, budget {budget} has methods {observed}; expected {expected_methods}."
            )

metric_values = raw_results[METRIC_COLUMNS].to_numpy(dtype=float)
if not np.isfinite(metric_values).all() or ((metric_values < 0) | (metric_values > 1)).any():
    raise AssertionError("Every reported metric must be finite and between 0 and 1.")

for (seed, budget), (labelled, unlabelled) in split_cache.items():
    if np.intersect1d(labelled, unlabelled).size or len(labelled) + len(unlabelled) != len(y_train):
        raise AssertionError(f"Invalid split for seed={seed}, budget={budget}.")

if RUN_ABLATION:
    after = ablation_results[ablation_results["threshold"].notna()]
    if not np.array_equal(
        after["passed_threshold"].to_numpy(dtype=int),
        after["pseudo_labels"].to_numpy(dtype=int),
    ):
        raise AssertionError("The ablation cap was not fully disabled.")
    if after["pseudo_labels"].nunique() == 1:
        warnings.warn(
            "All ablation thresholds still selected the same count. Report this null result explicitly.",
            RuntimeWarning,
        )

reference_values = np.asarray([LAB1_REFERENCE[column] for column in METRIC_COLUMNS], dtype=float)
if not np.isfinite(reference_values).all() or ((reference_values < 0) | (reference_values > 1)).any():
    raise AssertionError("The Lab 1 reference must contain all five valid metrics.")

manifest = {
    "data_mode": ACTIVE_DATA_MODE,
    "lab1_split_path": str(LAB1_SPLITS),
    "lab1_split_sha256": LAB1_SPLIT_SHA256,
    "expected_lab1_sha256": EXPECTED_LAB1_SHA256,
    "lab1_metrics_path": str(LAB1_METRICS_PATH),
    "lab1_reference": LAB1_REFERENCE,
    "backend": ACTIVE_BACKEND,
    "seeds": list(SEEDS),
    "budgets": list(BUDGETS),
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "ssl_rounds": SSL_ROUNDS,
    "pseudo_weight": PSEUDO_WEIGHT,
    "main_per_class_ratio_per_round": PSEUDO_TO_TRUE_RATIO_PER_ROUND,
    "ablation_budget": ABLATION_BUDGET,
    "ablation_thresholds": list(ABLATION_THRESHOLDS),
    "ablation_class_cap_enabled": False,
    "train_rows": len(y_train),
    "validation_rows": len(y_val),
    "test_rows": len(y_test),
    "features": X_train.shape[1],
    "cpu_threads": CPU_THREADS,
    "versions": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
    },
}
with open(OUTPUT_DIR / "run_manifest.json", "w", encoding="utf-8") as stream:
    json.dump(manifest, stream, indent=2)

print("ALL AUTOMATED CHECKS PASSED FOR THE VERIFIED LAB 1 SPLIT")
print("The manifest records the exact split digest and experiment configuration.")


## 15. How to run and report the results

1. Install Python 3.10+ and `numpy`, `pandas`, `scikit-learn`, `matplotlib`, `threadpoolctl`, `joblib`, and Jupyter.
2. Keep Lab 1's `metrics.py` at one of the configured paths, or update `LAB1_METRICS_CANDIDATES` to its real location.
3. Export the exact Lab 1 arrays to `data/lab1_splits.npz` using Section 3. Do not recreate a split from the eight raw CSVs.
4. Restart the kernel and choose **Run All**. Do not submit unless Section 14 prints the all-checks-passed message.
5. Copy Table 1, Table 2, and Figure 1 from the final run into the 2-3 page report. Discuss macro-F1, FAR, and attack recall—not accuracy alone.
6. Explain the main cap as a safety choice, use the pre-cap counts and audit precision to show when it dominated, and describe any SSL loss as a result rather than hiding it.
7. Add an honest contribution line for the Lab 2 work.

**Suggested discussion structure:** why labels are scarce; exact data/split and methods; Table 1/Figure 1 findings; threshold ablation; confirmation-bias or feature-view failures; operational meaning of FAR; contribution statement.

**Contribution statement:** `[Name] implemented and ran ...; [Name] analysed ... and prepared ... .`

### References

- Lee, D.-H. (2013). *Pseudo-Label: The Simple and Efficient Semi-Supervised Learning Method for Deep Neural Networks.* ICML Workshop.
- Van Engelen, J. E., & Hoos, H. H. (2020). A survey on semi-supervised learning. *Machine Learning, 109*(2), 373-440.
- scikit-learn documentation: semi-supervised learning and `HistGradientBoostingClassifier`.
- CICIDS2017: Canadian Institute for Cybersecurity.
